In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime

MovieLens, fichier movies, dataframe MVL_MOVIES


16 films n'ont pas de genre. Les lignes ne sont pas supprimées

Le champ MovieId est unique, mais pas title. Envisager un dédoublonnage pour mettre le title en index ?

In [25]:
# chargement du fichier
mvl_mv = pd.read_csv('data/ml-20m/movies.csv')

# Remplacement du genre (no genres listed) par NaN
mvl_mv['genres'] = mvl_mv['genres'].replace('(no genres listed)', np.nan)

# Séparation des genres en listes
mvl_movies = mvl_mv.join(
    mvl_mv['genres'].str.get_dummies(sep='|').add_prefix('genre_')
)

mvl_movies = mvl_movies.drop(columns=['genres'])

display(mvl_movies.head(2))

mvl_movies.info()

,movieId,title,genre_Action,genre_Adventure,genre_Animation,genre_Children,genre_Comedy,genre_Crime,genre_Documentary,genre_Drama,...,genre_Film-Noir,genre_Horror,genre_IMAX,genre_Musical,genre_Mystery,genre_Romance,genre_Sci-Fi,genre_Thriller,genre_War,genre_Western
0,1,Toy Story (1995),0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


<class 'pandas.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   movieId            27278 non-null  int64
 1   title              27278 non-null  str  
 2   genre_Action       27278 non-null  int64
 3   genre_Adventure    27278 non-null  int64
 4   genre_Animation    27278 non-null  int64
 5   genre_Children     27278 non-null  int64
 6   genre_Comedy       27278 non-null  int64
 7   genre_Crime        27278 non-null  int64
 8   genre_Documentary  27278 non-null  int64
 9   genre_Drama        27278 non-null  int64
 10  genre_Fantasy      27278 non-null  int64
 11  genre_Film-Noir    27278 non-null  int64
 12  genre_Horror       27278 non-null  int64
 13  genre_IMAX         27278 non-null  int64
 14  genre_Musical      27278 non-null  int64
 15  genre_Mystery      27278 non-null  int64
 16  genre_Romance      27278 non-null  int64
 17  genre_Sci-Fi       2727

MovieLens, fichier ratings, dataframe MVL_RATINGS

In [40]:
# chargement du fichier
mvl_ratings = pd.read_csv('data/ml-20m/ratings.csv')

mvl_ratings = mvl_ratings.drop(columns=['timestamp'])

display(mvl_ratings.head(2))

mvl_ratings.info(show_counts=True)

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5


<class 'pandas.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 3 columns):
 #   Column   Non-Null Count     Dtype  
---  ------   --------------     -----  
 0   userId   20000263 non-null  int64  
 1   movieId  20000263 non-null  int64  
 2   rating   20000263 non-null  float64
dtypes: float64(1), int64(2)
memory usage: 457.8 MB


MovieLens, fichier links, dataframe MVL_LINKS

In [41]:
# chargement du fichier
mvl_links = pd.read_csv('data/ml-20m/links.csv')

mvl_links = mvl_links.drop(columns=['tmdbId'])

display(mvl_links.head(2))

mvl_links.info(show_counts=True)

,movieId,imdbId
0,1,114709
1,2,113497


<class 'pandas.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   movieId  27278 non-null  int64
 1   imdbId   27278 non-null  int64
dtypes: int64(2)
memory usage: 426.3 KB


MovieLens, fichiers genome_scores et genome_tags , dataframe MVL_TAGS_SCORES

In [42]:
# chargement des fichiers
genome_tags = pd.read_csv('data/ml-20m/genome-tags.csv')
genome_scores = pd.read_csv('data/ml-20m/genome-scores.csv')

# inner join sur le champ tagId
mvl_tags_scores = genome_scores.merge(genome_tags, on='tagId', how='inner')

display(mvl_tags_scores.head(2))
mvl_tags_scores.info(show_counts=True)


,movieId,tagId,relevance,tag
0,1,1,0.025,007
1,1,2,0.025,007 (series)


<class 'pandas.DataFrame'>
RangeIndex: 11709768 entries, 0 to 11709767
Data columns (total 4 columns):
 #   Column     Non-Null Count     Dtype  
---  ------     --------------     -----  
 0   movieId    11709768 non-null  int64  
 1   tagId      11709768 non-null  int64  
 2   relevance  11709768 non-null  float64
 3   tag        11709768 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 357.4 MB


Récapitulatif des datasets MovieLens

mvl_movies : liste des films, une ligne par film, une colonne par genre avec valeur binaire

mvl_ratings : note des différents utilisateurs

mvl_links : correspondance entre l'identifiant movieLens et IMDB

mvl_tags_scores : pertinence des films pour chacun des 1128 tags

*************************************************************

IMDB, fichier name.basics, dataframe personnes

In [48]:
# chargement du fichier
personnes = pd.read_csv('data/imdb/name.basics.tsv', sep='\t',na_values='\\N')

# conversion du champ nconst (identifiant de personne) en entier
personnes["nconst"] = personnes["nconst"].str[2:].astype(int)

# suppression des colonnes inutiles
personnes = personnes.drop(columns=['birthYear', 'deathYear', 'primaryProfession', 'knownForTitles'])

# suppression des personnes qui n'ont pas de nom
personnes = personnes.dropna(subset=['primaryName'])

display(personnes.head(2))
personnes.info(show_counts=True)

,nconst,primaryName
0,1,Fred Astaire
1,2,Lauren Bacall


<class 'pandas.DataFrame'>
Index: 15225264 entries, 0 to 15225342
Data columns (total 2 columns):
 #   Column       Non-Null Count     Dtype
---  ------       --------------     -----
 0   nconst       15225264 non-null  int64
 1   primaryName  15225264 non-null  str  
dtypes: int64(1), str(1)
memory usage: 348.5 MB


IMDB, fichier title.basics, dataframe imdb_movies

Le genre du film est renseigné dans IMDB et movieLens. Nous ne gardons que celui de MovieLens

Il y a des NA dans les champs primaryTitle, startYear et runtimeMinutes. Nous apprécierons après le merge avec les données de movielens

In [69]:
# chargement du fichier
title_basics = pd.read_csv('data/imdb/title.basics.tsv', sep='\t',na_values='\\N')

# conversion du champ tconst (identifiant de titre/film) en entier
title_basics["tconst"] = title_basics["tconst"].str[2:].astype(int)

# suppression des colonnes inutiles
title_basics = title_basics.drop(columns=['originalTitle', 'endYear', 'genres'])

# filtre sur les films
# title_basics = title_basics[title_basics["titleType"].isin(["movie", "tvMovie"])]
#imdb_movies = title_basics[title_basics["titleType"].isin(["movie", "tvMovie"])]
title_basics = title_basics[title_basics["titleType"].isin(["movie"])]

/var/folders/43/j4zwb6d13hd0m383lzvlyljr0000gn/T/ipykernel_75029/367708964.py:2: DtypeWarning: Columns (0: runtimeMinutes) have mixed types. Specify dtype option on import or set low_memory=False.
  title_basics = pd.read_csv('data/imdb/title.basics.tsv', sep='\t',na_values='\\N')


In [73]:
display(title_basics.head(2))
title_basics.info()
print("\nValeurs manquantes : ", end="\n\n")
display(pd.DataFrame({
    "nb_na": title_basics.isna().sum(),
    "pct_na": title_basics.isna().mean() * 100
}))
print("\nValeurs uniques : ", end="\n\n")
print(title_basics.nunique())

,tconst,titleType,primaryTitle,isAdult,startYear,runtimeMinutes
8,9,movie,Miss Jerry,0,1894.0,45.0
144,147,movie,The Corbett-Fitzsimmons Fight,0,1897.0,100.0


<class 'pandas.DataFrame'>
Index: 742654 entries, 8 to 12420916
Data columns (total 6 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          742654 non-null  int64  
 1   titleType       742654 non-null  str    
 2   primaryTitle    742651 non-null  str    
 3   isAdult         742654 non-null  int64  
 4   startYear       631464 non-null  float64
 5   runtimeMinutes  468388 non-null  object 
dtypes: float64(1), int64(2), object(1), str(2)
memory usage: 39.7+ MB

Valeurs manquantes : 



,nb_na,pct_na
tconst,0,0.000000
titleType,0,0.000000
primaryTitle,3,0.000404
isAdult,0,0.000000
startYear,111190,14.971979
runtimeMinutes,274266,36.930522



Valeurs uniques : 

tconst            742654
titleType              1
primaryTitle      637239
isAdult                2
startYear            137
runtimeMinutes       792
dtype: int64


IMDB, fichier title.ratings, dataframe imdb_ratings

In [76]:
# chargement du fichier
title_ratings = pd.read_csv('data/imdb/title.ratings.tsv', sep='\t',na_values='\\N')

# conversion du champ tconst (identifiant de titre/film) en entier
title_ratings["tconst"] = title_ratings["tconst"].str[2:].astype(int)

display(title_ratings.head(2))
title_ratings.info()

,tconst,averageRating,numVotes
0,1,5.7,2201
1,2,5.5,312


<class 'pandas.DataFrame'>
RangeIndex: 1658155 entries, 0 to 1658154
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1658155 non-null  int64  
 1   averageRating  1658155 non-null  float64
 2   numVotes       1658155 non-null  int64  
dtypes: float64(1), int64(2)
memory usage: 38.0 MB


In [77]:
display(pd.DataFrame({
    "nb_na": title_ratings.isna().sum(),
    "pct_na": title_ratings.isna().mean() * 100
}))

,nb_na,pct_na
tconst,0,0.0
averageRating,0,0.0
numVotes,0,0.0
